# Round 5 — Purification Pebbles

Products:
- `PEBBLES_XS`
- `PEBBLES_S`
- `PEBBLES_M`
- `PEBBLES_L`
- `PEBBLES_XL`

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.dirname(os.getcwd())), ""))
from plotter import Plotter, LogVisualizer
import pandas as pd
import plotly.graph_objects as go

DATA_DIR = "../../data/round5"
days = [2, 3, 4]
PRODUCTS = [
    "PEBBLES_XS",
    "PEBBLES_S",
    "PEBBLES_M",
    "PEBBLES_L",
    "PEBBLES_XL",
]

In [2]:
raw_prices = pd.concat(
    [pd.read_csv(f"{DATA_DIR}/prices_round_5_day_{d}.csv", delimiter=";") for d in days],
    ignore_index=True,
)
all_prices = raw_prices[raw_prices["product"].isin(PRODUCTS)].reset_index(drop=True)

frames = {p: all_prices[all_prices["product"] == p].reset_index(drop=True) for p in PRODUCTS}
for p, df in frames.items():
    print(f"{p}: {len(df)} rows")

PEBBLES_XS: 30000 rows
PEBBLES_S: 30000 rows
PEBBLES_M: 30000 rows
PEBBLES_L: 30000 rows
PEBBLES_XL: 30000 rows


In [3]:
raw_trades = pd.concat(
    [pd.read_csv(f"{DATA_DIR}/trades_round_5_day_{d}.csv", delimiter=";")
       .assign(day=d)
     for d in days],
    ignore_index=True,
).rename(columns={"symbol": "product"})

all_trades = raw_trades[raw_trades["product"].isin(PRODUCTS)].reset_index(drop=True)
all_trades = all_trades.merge(all_prices, on=["day", "timestamp", "product"], how="left")

trades_by_product = {p: all_trades[all_trades["product"] == p].reset_index(drop=True) for p in PRODUCTS}
for p, df in trades_by_product.items():
    print(f"{p}: {len(df)} trades")

PEBBLES_XS: 644 trades
PEBBLES_S: 644 trades
PEBBLES_M: 644 trades
PEBBLES_L: 644 trades
PEBBLES_XL: 644 trades


## Spread distribution

In [4]:
for p, df in frames.items():
    spread = df["ask_price_1"] - df["bid_price_1"]
    print(f"\n{p} spread:")
    print(spread.describe().to_string())


PEBBLES_XS spread:
count    30000.000000
mean         9.744867
std          1.953126
min          3.000000
25%          8.000000
50%          9.000000
75%         11.000000
max         14.000000

PEBBLES_S spread:
count    30000.000000
mean        11.551533
std          1.453254
min          4.000000
25%         11.000000
50%         12.000000
75%         12.000000
max         14.000000

PEBBLES_M spread:
count    30000.000000
mean        13.120900
std          1.436971
min          5.000000
25%         13.000000
50%         13.000000
75%         14.000000
max         15.000000

PEBBLES_L spread:
count    30000.000000
mean        13.020500
std          1.387952
min          5.000000
25%         13.000000
50%         13.000000
75%         14.000000
max         15.000000

PEBBLES_XL spread:
count    30000.000000
mean        16.630767
std          2.568849
min          5.000000
25%         16.000000
50%         17.000000
75%         18.000000
max         22.000000


## Mid price overview

In [24]:
DAY_OFFSET = 1_000_000
fig = go.Figure()
for p, df in frames.items():
    df = df.sort_values(["day", "timestamp"]).copy()
    df["global_ts"] = df["timestamp"] + (df["day"] - df["day"].min()) * DAY_OFFSET
    fig.add_trace(go.Scattergl(x=df["global_ts"], y=df["mid_price"], mode="lines", name=p))
fig.update_layout(title="Pebbles — mid prices", xaxis_title="global timestamp", yaxis_title="mid price")
fig.show(renderer="browser")

## Per-product orderbook

In [16]:
plt = Plotter(
    [f"{DATA_DIR}/prices_round_5_day_{d}.csv" for d in days],
    [f"{DATA_DIR}/trades_round_5_day_{d}.csv" for d in days],
)
# restrict the plotter to the pebbles products only
plt.prices = plt.prices[plt.prices["product"].isin(PRODUCTS)].reset_index(drop=True)
plt.trades = plt.trades[plt.trades["product"].isin(PRODUCTS)].reset_index(drop=True)
plt.products = [p for p in PRODUCTS if p in set(plt.prices["product"].unique())]

plt.visualize_orderbook(renderer="browser")
#spreads are decreasing for the smaller ones over time and increase for the larger ones over time, stays the same for the medium

interactive(children=(Dropdown(description='Product:', options=('PEBBLES_XS', 'PEBBLES_S', 'PEBBLES_M', 'PEBBL…

## Cointegration tests (Engle-Granger)

In [7]:
from statsmodels.tsa.stattools import coint
from itertools import combinations
import numpy as np

# align mid prices on (day, timestamp) so all series have the same index
mid_wide = (
    all_prices[["day", "timestamp", "product", "mid_price"]]
    .pivot_table(index=["day", "timestamp"], columns="product", values="mid_price")
    .sort_index()
    .dropna()
)
print(f"aligned rows: {len(mid_wide)}")

results = []
for a, b in combinations(PRODUCTS, 2):
    if a not in mid_wide.columns or b not in mid_wide.columns:
        continue
    t_stat, p_value, crit = coint(mid_wide[a], mid_wide[b])
    results.append({
        "pair": f"{a} | {b}",
        "t_stat": t_stat,
        "p_value": p_value,
        "crit_1%": crit[0],
        "crit_5%": crit[1],
        "crit_10%": crit[2],
        "cointegrated_5%": p_value < 0.05,
    })

coint_df = pd.DataFrame(results).sort_values("p_value").reset_index(drop=True)
coint_df

aligned rows: 30000


,pair,t_stat,p_value,crit_1%,crit_5%,crit_10%,cointegrated_5%
0,PEBBLES_XS | PEBBLES_S,-2.714892,0.194268,-3.896805,-3.336334,-3.044591,False
1,PEBBLES_XS | PEBBLES_M,-2.684356,0.205256,-3.896805,-3.336334,-3.044591,False
2,PEBBLES_S | PEBBLES_XL,-2.613309,0.231502,-3.896805,-3.336334,-3.044591,False
3,PEBBLES_S | PEBBLES_M,-2.602466,0.235838,-3.896805,-3.336334,-3.044591,False
4,PEBBLES_M | PEBBLES_XL,-2.372667,0.337908,-3.896805,-3.336334,-3.044591,False
5,PEBBLES_M | PEBBLES_L,-1.968895,0.544704,-3.896805,-3.336334,-3.044591,False
6,PEBBLES_XS | PEBBLES_XL,-1.934674,0.562323,-3.896805,-3.336334,-3.044591,False
7,PEBBLES_L | PEBBLES_XL,-1.810306,0.624703,-3.896805,-3.336334,-3.044591,False
8,PEBBLES_S | PEBBLES_L,-1.578123,0.730035,-3.896805,-3.336334,-3.044591,False
9,PEBBLES_XS | PEBBLES_L,-1.357892,0.811582,-3.896805,-3.336334,-3.044591,False


## All products combined view

All 5 products on one chart — mid + bids + asks + trades. Click legend entries to toggle individual traces, or double-click to isolate. Traces are grouped by product so toggling a group hides all of its lines/markers.

In [ ]:
import plotly.colors as pc

DAY_OFFSET = 1_000_000
colors = pc.qualitative.Plotly  # one color per product

fig = go.Figure()

for i, product in enumerate(PRODUCTS):
    color = colors[i % len(colors)]
    df = frames[product].sort_values(["day", "timestamp"]).copy()
    df["global_ts"] = df["timestamp"] + (df["day"] - df["day"].min()) * DAY_OFFSET

    # mid price (visible by default)
    fig.add_trace(go.Scattergl(
        x=df["global_ts"], y=df["mid_price"],
        mode="lines", name=f"{product} mid",
        legendgroup=product, line=dict(color=color, width=2),
    ))

    # bid/ask levels (hidden by default — click legend to show)
    for lvl in range(1, 4):
        fig.add_trace(go.Scattergl(
            x=df["global_ts"], y=df[f"bid_price_{lvl}"],
            mode="lines", name=f"{product} bid_{lvl}",
            legendgroup=product, visible="legendonly",
            line=dict(color=color, width=1, dash="dash"),
        ))
        fig.add_trace(go.Scattergl(
            x=df["global_ts"], y=df[f"ask_price_{lvl}"],
            mode="lines", name=f"{product} ask_{lvl}",
            legendgroup=product, visible="legendonly",
            line=dict(color=color, width=1, dash="dot"),
        ))

    # trades (visible by default, smaller markers)
    tr = trades_by_product[product].sort_values(["day", "timestamp"]).copy()
    if not tr.empty:
        tr["global_ts"] = tr["timestamp"] + (tr["day"] - tr["day"].min()) * DAY_OFFSET
        sizes = 4 + 8 * (tr["quantity"] / max(tr["quantity"].max(), 1))
        fig.add_trace(go.Scattergl(
            x=tr["global_ts"], y=tr["price"],
            mode="markers", name=f"{product} trades",
            legendgroup=product,
            marker=dict(color=color, size=sizes, symbol="circle", opacity=0.6,
                        line=dict(width=0.5, color="white")),
            customdata=tr["quantity"],
            hovertemplate=f"{product}<br>price: %{{y}}<br>qty: %{{customdata}}<br>ts: %{{x}}<extra></extra>",
        ))

fig.update_layout(
    title="Pebbles — all products (mid + trades; bids/asks hidden, click legend to show)",
    xaxis_title="global timestamp",
    yaxis_title="price",
    hovermode="closest",
    legend=dict(groupclick="togglegroup"),
    height=700,
)
fig.show(renderer="browser")

## Trade-flow alignment across products

Are trades synchronized across all 5 pebble sizes (same timestamps, quantities, sides)?

In [8]:
# check if (timestamp, quantity) of trades are identical across all 5 products
ref = PRODUCTS[0]
ref_df = trades_by_product[ref][["timestamp", "quantity"]].sort_values(["timestamp", "quantity"]).reset_index(drop=True)

print(f"reference: {ref} — {len(ref_df)} trades")
for p in PRODUCTS[1:]:
    df = trades_by_product[p][["timestamp", "quantity"]].sort_values(["timestamp", "quantity"]).reset_index(drop=True)
    same_len = len(df) == len(ref_df)
    same_ts = same_len and (df["timestamp"].values == ref_df["timestamp"].values).all()
    same_qty = same_len and (df["quantity"].values == ref_df["quantity"].values).all()
    print(f"  {p}: rows={len(df)} same_len={same_len} same_ts={same_ts} same_qty={same_qty}")

reference: PEBBLES_XS — 644 trades
  PEBBLES_S: rows=644 same_len=True same_ts=True same_qty=True
  PEBBLES_M: rows=644 same_len=True same_ts=True same_qty=True
  PEBBLES_L: rows=644 same_len=True same_ts=True same_qty=True
  PEBBLES_XL: rows=644 same_len=True same_ts=True same_qty=True


In [9]:
# classify each trade by side using merged bid/ask, then compare across products
def classify(df):
    side = pd.Series("mid", index=df.index)
    side[df["price"] >= df["ask_price_1"]] = "buy"
    side[df["price"] <= df["bid_price_1"]] = "sell"
    return df.assign(side=side)[["timestamp", "side"]].sort_values("timestamp").reset_index(drop=True)

ref = PRODUCTS[0]
ref_df = classify(trades_by_product[ref])
print(f"reference: {ref} — sides: {ref_df['side'].value_counts().to_dict()}")

for p in PRODUCTS[1:]:
    df = classify(trades_by_product[p])
    same_len = len(df) == len(ref_df)
    same_ts = same_len and (df["timestamp"].values == ref_df["timestamp"].values).all()
    same_side = same_len and (df["side"].values == ref_df["side"].values).all()
    mismatches = 0 if same_side else int((df["side"].values != ref_df["side"].values).sum()) if same_len else "n/a"
    print(f"  {p}: same_ts={same_ts} same_side={same_side} mismatches={mismatches}")

reference: PEBBLES_XS — sides: {'sell': 323, 'buy': 321}
  PEBBLES_S: same_ts=True same_side=True mismatches=0
  PEBBLES_M: same_ts=True same_side=True mismatches=0
  PEBBLES_L: same_ts=True same_side=True mismatches=0
  PEBBLES_XL: same_ts=True same_side=True mismatches=0


## Counterparty PnL — what does the market-taker make?

Imagine one trader did every trade across all 5 pebble products. What's their PnL?

In [10]:
def classify(df):
    side = pd.Series("mid", index=df.index)
    side[df["price"] >= df["ask_price_1"]] = "buy"
    side[df["price"] <= df["bid_price_1"]] = "sell"
    return df.assign(side=side)

per_product_pnl = {}
total_pnl = 0.0
for p in PRODUCTS:
    df = classify(trades_by_product[p])
    cash = (df.loc[df["side"] == "sell", "price"] * df.loc[df["side"] == "sell", "quantity"]).sum() \
         - (df.loc[df["side"] == "buy",  "price"] * df.loc[df["side"] == "buy",  "quantity"]).sum()
    net_qty = df.loc[df["side"] == "buy", "quantity"].sum() - df.loc[df["side"] == "sell", "quantity"].sum()
    last_mid = frames[p]["mid_price"].dropna().iloc[-1]
    pnl = cash + net_qty * last_mid
    per_product_pnl[p] = pnl
    total_pnl += pnl
    print(f"{p}: cash={cash:+,.2f}  net_qty={net_qty:+d}  last_mid={last_mid:.2f}  pnl={pnl:+,.2f}")

print(f"\nTOTAL PnL across all 5 products: {total_pnl:+,.2f}")

PEBBLES_XS: cash=+195,390.00  net_qty=-17  last_mid=6038.00  pnl=+92,744.00
PEBBLES_S: cash=+138,837.00  net_qty=-17  last_mid=8066.50  pnl=+1,706.50
PEBBLES_M: cash=+181,323.00  net_qty=-17  last_mid=10702.00  pnl=-611.00
PEBBLES_L: cash=+125,843.00  net_qty=-17  last_mid=9126.00  pnl=-29,299.00
PEBBLES_XL: cash=+135,059.00  net_qty=-17  last_mid=16068.00  pnl=-138,097.00

TOTAL PnL across all 5 products: -73,556.50


In [11]:
# total buy and sell quantities across all 5 products
def classify(df):
    side = pd.Series("mid", index=df.index)
    side[df["price"] >= df["ask_price_1"]] = "buy"
    side[df["price"] <= df["bid_price_1"]] = "sell"
    return df.assign(side=side)

total_buy_qty = 0
total_sell_qty = 0
for p in PRODUCTS:
    df = classify(trades_by_product[p])
    buy_qty = int(df.loc[df["side"] == "buy", "quantity"].sum())
    sell_qty = int(df.loc[df["side"] == "sell", "quantity"].sum())
    total_buy_qty += buy_qty
    total_sell_qty += sell_qty
    print(f"{p}: buys={buy_qty}  sells={sell_qty}")

print(f"\nTotal buy quantity:  {total_buy_qty}")
print(f"Total sell quantity: {total_sell_qty}")

PEBBLES_XS: buys=1133  sells=1150
PEBBLES_S: buys=1133  sells=1150
PEBBLES_M: buys=1133  sells=1150
PEBBLES_L: buys=1133  sells=1150
PEBBLES_XL: buys=1133  sells=1150

Total buy quantity:  5665
Total sell quantity: 5750


## Spread at trade time

In [12]:
# spread (ask_1 - bid_1) at the moment each trade occurred, per product
for p in PRODUCTS:
    df = trades_by_product[p].copy()
    df["spread"] = df["ask_price_1"] - df["bid_price_1"]
    print(f"\n{p}:")
    print(df["spread"].value_counts().sort_index().to_string())
    print(f"  mean: {df['spread'].mean():.2f}  median: {df['spread'].median():.2f}  n_trades: {len(df)}")


PEBBLES_XS:
spread
3       1
4       3
5       4
6       3
7      37
8     146
9     134
10     82
11     83
12     89
13     55
14      7
  mean: 9.78  median: 9.00  n_trades: 644

PEBBLES_S:
spread
4       2
6       6
7       4
9      13
10     78
11    121
12    268
13    142
14     10
  mean: 11.65  median: 12.00  n_trades: 644

PEBBLES_M:
spread
6       4
7       7
8       1
11     17
12    114
13    215
14    229
15     57
  mean: 13.19  median: 13.00  n_trades: 644

PEBBLES_L:
spread
5       1
6       2
7       6
8       3
11      8
12    103
13    276
14    211
15     34
  mean: 13.14  median: 13.00  n_trades: 644

PEBBLES_XL:
spread
6       1
7       3
8       2
9       3
10      2
11      1
12     17
13     32
14     56
15     31
16     97
17    174
18    119
19     26
20     41
21     34
22      5
  mean: 16.71  median: 17.00  n_trades: 644


## Mid-price first-difference distributions

Compares per-step changes in mid/bid/ask against a fitted normal — useful for sanity-checking the random-walk assumption.

In [17]:
import plotly.graph_objects as go
import plotly.colors as pc
from scipy import stats
import numpy as np

colors = pc.qualitative.Plotly

def plot_diff_distribution(col, title):
    fig = go.Figure()
    for i, p in enumerate(PRODUCTS):
        diffs = frames[p].groupby("day")[col].diff().dropna()
        if diffs.empty:
            continue
        color = colors[i % len(colors)]
        mu, sigma = diffs.mean(), diffs.std()

        fig.add_trace(go.Histogram(
            x=diffs, name=f"{p} hist",
            legendgroup=p, marker_color=color,
            opacity=0.5, nbinsx=80, histnorm="probability density",
        ))

        xs = np.linspace(diffs.min(), diffs.max(), 400)
        pdf = stats.norm.pdf(xs, loc=mu, scale=sigma)
        fig.add_trace(go.Scatter(
            x=xs, y=pdf, mode="lines",
            name=f"{p} N({mu:+.3f}, {sigma:.3f})",
            legendgroup=p, line=dict(color=color, width=2),
        ))

        print(f"{p}: mean={mu:+.4f}  std={sigma:.4f}  n={len(diffs)}")

    fig.update_layout(
        barmode="overlay",
        title=title,
        xaxis_title=f"{col}_t - {col}_{{t-1}}",
        yaxis_title="density",
        legend=dict(groupclick="togglegroup"),
    )
    fig.show(renderer="browser")

plot_diff_distribution("mid_price", "Mid-price first-difference distribution + normal fit")

PEBBLES_XS: mean=-0.1326  std=15.0524  n=29997
PEBBLES_S: mean=-0.0651  std=15.0203  n=29997
PEBBLES_M: mean=+0.0230  std=15.1336  n=29997
PEBBLES_L: mean=-0.0297  std=15.0255  n=29997
PEBBLES_XL: mean=+0.2046  std=30.3142  n=29997


In [18]:
plot_diff_distribution("bid_price_1", "Best-bid first-difference distribution + normal fit")

PEBBLES_XS: mean=-0.1326  std=15.0489  n=29997
PEBBLES_S: mean=-0.0651  std=15.0251  n=29997
PEBBLES_M: mean=+0.0230  std=15.1391  n=29997
PEBBLES_L: mean=-0.0297  std=15.0343  n=29997
PEBBLES_XL: mean=+0.2045  std=30.3058  n=29997


In [ ]:
plot_diff_distribution("ask_price_1", "Best-ask first-difference distribution + normal fit")

## Level-1 vs level-2 mid and spread

In [15]:
# normal mid (level 1) vs wall mid (level 2), and spread distributions for both
for p in PRODUCTS:
    df = frames[p].copy()
    df["mid_1"] = (df["bid_price_1"] + df["ask_price_1"]) / 2
    df["mid_2"] = (df["bid_price_2"] + df["ask_price_2"]) / 2
    df["spread_1"] = df["ask_price_1"] - df["bid_price_1"]
    df["spread_2"] = df["ask_price_2"] - df["bid_price_2"]

    print(f"\n=== {p} ===")
    print(f"normal mid (lvl 1): mean={df['mid_1'].mean():.2f}  std={df['mid_1'].std():.2f}  n={df['mid_1'].notna().sum()}")
    print(f"wall mid   (lvl 2): mean={df['mid_2'].mean():.2f}  std={df['mid_2'].std():.2f}  n={df['mid_2'].notna().sum()}")

    print("\nspread_1 (ask_1 - bid_1) counts:")
    print(df["spread_1"].value_counts().sort_index().to_string())
    print("\nspread_2 (ask_2 - bid_2) counts:")
    print(df["spread_2"].value_counts().sort_index().to_string())


=== PEBBLES_XS ===
normal mid (lvl 1): mean=7404.64  std=1449.55  n=30000
wall mid   (lvl 2): mean=7532.48  std=1428.17  n=27775

spread_1 (ask_1 - bid_1) counts:
spread_1
3       72
4      248
5      285
6      185
7     1219
8     6926
9     7018
10    3220
11    3752
12    4179
13    2664
14     232

spread_2 (ask_2 - bid_2) counts:
spread_2
8.0       38
9.0     1347
10.0    5774
11.0    5765
12.0    2928
13.0    2772
14.0    3377
15.0    3174
16.0    2311
17.0     289

=== PEBBLES_S ===
normal mid (lvl 1): mean=8932.36  std=833.28  n=30000
wall mid   (lvl 2): mean=8932.37  std=833.28  n=30000

spread_1 (ask_1 - bid_1) counts:
spread_1
4        43
5       238
6       398
7       169
8         5
9       773
10     3621
11     6013
12    12110
13     5877
14      753

spread_2 (ask_2 - bid_2) counts:
spread_2
10.0       17
11.0      611
12.0     2364
13.0     4846
14.0     6323
15.0    10405
16.0     4494
17.0      940

=== PEBBLES_M ===
normal mid (lvl 1): mean=10263.24  std=687.82 

In [19]:
# for each product, when spread_1 is low (<= some threshold), what does spread_2 look like?
LOW_SPREAD_THRESH = 7

for p in PRODUCTS:
    df = frames[p].copy()
    df["spread_1"] = df["ask_price_1"] - df["bid_price_1"]
    df["spread_2"] = df["ask_price_2"] - df["bid_price_2"]
    low = df[df["spread_1"] <= LOW_SPREAD_THRESH]

    print(f"\n=== {p} ===  (rows with spread_1 <= {LOW_SPREAD_THRESH}: {len(low)} / {len(df)})")
    if low.empty:
        continue

    ct = pd.crosstab(low["spread_1"], low["spread_2"], dropna=False)
    print(ct.to_string())

    s2 = low["spread_2"]
    print(f"\nspread_2 when spread_1 <= {LOW_SPREAD_THRESH}: "
          f"mean={s2.mean():.2f}  median={s2.median():.2f}  "
          f"missing={s2.isna().sum()}  n={s2.notna().sum()}")


=== PEBBLES_XS ===  (rows with spread_1 <= 7: 2009 / 30000)
spread_2  8.0   9.0   10.0  11.0  12.0  13.0  14.0  15.0  NaN 
spread_1                                                      
3           23    49     0     0     0     0     0     0     0
4           15   100    81    25     6     0     0     0    21
5            0    49    88    35    43    34    15     0    21
6            0     0     0    17    35    65    50    18     0
7            0  1149     0     0     0    10    32    18    10

spread_2 when spread_1 <= 7: mean=9.86  median=9.00  missing=52  n=1957

=== PEBBLES_S ===  (rows with spread_1 <= 7: 848 / 30000)
spread_2  10.0  11.0  12.0  13.0  14.0  15.0  16.0
spread_1                                          
4            7    21    15     0     0     0     0
5           10    37    77    66    48     0     0
6            0    27    64   135   129    39     4
7            0     0     0    72    56    41     0

spread_2 when spread_1 <= 7: mean=13.03  median=13.00  miss

In [20]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

DAY_OFFSET = 1_000_000

def classify(df):
    side = pd.Series("mid", index=df.index)
    side[df["price"] >= df["ask_price_1"]] = "buy"
    side[df["price"] <= df["bid_price_1"]] = "sell"
    return df.assign(side=side)

fig = make_subplots(
    rows=len(PRODUCTS), cols=1, shared_xaxes=True,
    subplot_titles=PRODUCTS, vertical_spacing=0.03,
)

for i, p in enumerate(PRODUCTS, start=1):
    # mid line
    df = frames[p].sort_values(["day", "timestamp"]).copy()
    df["global_ts"] = df["timestamp"] + (df["day"] - df["day"].min()) * DAY_OFFSET
    fig.add_trace(go.Scattergl(
        x=df["global_ts"], y=df["mid_price"],
        mode="lines", name=f"{p} mid",
        legendgroup=p, line=dict(color="lightgray", width=1),
        showlegend=(i == 1),
    ), row=i, col=1)

    # trades, classified by side
    tr = classify(trades_by_product[p]).sort_values(["day", "timestamp"]).copy()
    tr["global_ts"] = tr["timestamp"] + (tr["day"] - tr["day"].min()) * DAY_OFFSET

    for side, color, symbol in [("buy", "#2ecc71", "triangle-up"),
                                ("sell", "#e74c3c", "triangle-down"),
                                ("mid", "#888888", "circle")]:
        sub = tr[tr["side"] == side]
        if sub.empty:
            continue
        fig.add_trace(go.Scattergl(
            x=sub["global_ts"], y=sub["price"],
            mode="markers", name=side,
            legendgroup=side, showlegend=(i == 1),
            marker=dict(color=color, size=6, symbol=symbol,
                        line=dict(width=0.5, color="white"), opacity=0.85),
            customdata=sub["quantity"],
            hovertemplate=f"{p}<br>{side}<br>price: %{{y}}<br>qty: %{{customdata}}<br>ts: %{{x}}<extra></extra>",
        ), row=i, col=1)

fig.update_layout(
    title="Pebbles — trades over time (buys vs sells)",
    height=300 * len(PRODUCTS),
    hovermode="closest",
)
fig.show(renderer="browser")


In [23]:
import numpy as np

def classify(df):
    side = pd.Series("mid", index=df.index)
    side[df["price"] >= df["ask_price_1"]] = "buy"
    side[df["price"] <= df["bid_price_1"]] = "sell"
    return df.assign(side=side)

DAY_OFFSET = 1_000_000

rows = []
for p in PRODUCTS:
    tr = classify(trades_by_product[p]).sort_values(["day", "timestamp"]).copy()
    tr["global_ts"] = tr["timestamp"] + (tr["day"] - tr["day"].min()) * DAY_OFFSET

    # gap between consecutive trades (any side)
    all_gaps = tr["global_ts"].diff().dropna()

    # gap between consecutive buys, and consecutive sells
    buy_gaps  = tr.loc[tr["side"] == "buy",  "global_ts"].diff().dropna()
    sell_gaps = tr.loc[tr["side"] == "sell", "global_ts"].diff().dropna()

    # for each buy, time until the next sell anywhere after it (and vice versa)
    sides_arr = tr["side"].values
    ts_arr    = tr["global_ts"].values
    b2s, s2b = [], []
    next_sell_ts = None
    next_buy_ts  = None
    for j in range(len(sides_arr) - 1, -1, -1):
        if sides_arr[j] == "buy" and next_sell_ts is not None:
            b2s.append(next_sell_ts - ts_arr[j])
        if sides_arr[j] == "sell" and next_buy_ts is not None:
            s2b.append(next_buy_ts - ts_arr[j])
        if sides_arr[j] == "sell": next_sell_ts = ts_arr[j]
        if sides_arr[j] == "buy":  next_buy_ts  = ts_arr[j]

    def stat(x):
        x = np.asarray(x, dtype=float)
        if len(x) == 0:
            return dict(n=0, mean=np.nan, median=np.nan, std=np.nan, min=np.nan, max=np.nan)
        return dict(n=len(x), mean=x.mean(), median=np.median(x),
                    std=x.std(), min=x.min(), max=x.max())

    rows.append({"product": p, "kind": "all_trades",   **stat(all_gaps)})
    rows.append({"product": p, "kind": "buy→buy",       **stat(buy_gaps)})
    rows.append({"product": p, "kind": "sell→sell",     **stat(sell_gaps)})
    rows.append({"product": p, "kind": "buy→next_sell", **stat(b2s)})
    rows.append({"product": p, "kind": "sell→next_buy", **stat(s2b)})

gap_df = pd.DataFrame(rows)
pd.set_option("display.float_format", lambda v: f"{v:,.1f}")
gap_df


,product,kind,n,mean,median,std,min,max
0,PEBBLES_XS,all_trades,643,"4,656.3","3,200.0","4,831.3",100.0,"41,900.0"
1,PEBBLES_XS,buy→buy,320,"9,325.3","6,700.0","9,049.8",100.0,"55,000.0"
2,PEBBLES_XS,sell→sell,322,"9,263.4","6,850.0","8,854.8",100.0,"73,700.0"
3,PEBBLES_XS,buy→next_sell,321,"8,857.6","6,300.0","10,333.9",100.0,"72,200.0"
4,PEBBLES_XS,sell→next_buy,320,"8,615.0","5,900.0","8,132.1",100.0,"49,100.0"
5,PEBBLES_S,all_trades,643,"4,656.3","3,200.0","4,831.3",100.0,"41,900.0"
6,PEBBLES_S,buy→buy,320,"9,325.3","6,700.0","9,049.8",100.0,"55,000.0"
7,PEBBLES_S,sell→sell,322,"9,263.4","6,850.0","8,854.8",100.0,"73,700.0"
8,PEBBLES_S,buy→next_sell,321,"8,857.6","6,300.0","10,333.9",100.0,"72,200.0"
9,PEBBLES_S,sell→next_buy,320,"8,615.0","5,900.0","8,132.1",100.0,"49,100.0"


In [25]:
import numpy as np
from scipy import stats

# Per-step drift and volatility of mid_price (timesteps are 100 apart).
# Diff within each day so we don't cross day boundaries.
rows = []
for p in PRODUCTS:
    diffs = frames[p].sort_values(["day", "timestamp"]).groupby("day")["mid_price"].diff().dropna()
    n     = len(diffs)
    mu    = diffs.mean()             # drift per 100-tick step
    sigma = diffs.std(ddof=1)        # vol per 100-tick step
    se    = sigma / np.sqrt(n)       # SE of the mean
    t     = mu / se
    p_val = 2 * (1 - stats.norm.cdf(abs(t)))

    steps_per_day = 9999             # 10,000 rows/day → 9,999 diffs
    rows.append({
        "product":          p,
        "n":                n,
        "mu_per_step":      mu,
        "sigma_per_step":   sigma,
        "SE(mu)":           se,
        "t_stat":           t,
        "p_value":          p_val,
        "drift_per_day":    mu * steps_per_day,
        "vol_per_day":      sigma * np.sqrt(steps_per_day),
        "signal_to_noise":  mu / sigma,   # μ/σ — how much drift per unit of vol
    })

drift_df = pd.DataFrame(rows)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
drift_df


,product,n,mu_per_step,sigma_per_step,SE(mu),t_stat,p_value,drift_per_day,vol_per_day,signal_to_noise
0,PEBBLES_XS,29997,-0.1326,15.0524,0.0869,-1.5261,0.1270,"-1,326.1667","1,505.1695",-0.0088
1,PEBBLES_S,29997,-0.0651,15.0203,0.0867,-0.7511,0.4526,-651.3333,"1,501.9579",-0.0043
2,PEBBLES_M,29997,0.0230,15.1336,0.0874,0.2631,0.7925,229.8333,"1,513.2819",0.0015
3,PEBBLES_L,29997,-0.0297,15.0255,0.0868,-0.3424,0.7321,-297.0000,"1,502.4792",-0.0020
4,PEBBLES_XL,29997,0.2046,30.3142,0.1750,1.1687,0.2425,"2,045.3333","3,031.2700",0.0067


In [26]:
import numpy as np
from scipy import stats

# Per-step drift and volatility of mid_price (timesteps are 100 apart).
# Diff within each day so we don't cross day boundaries.
rows = []
for p in PRODUCTS:
    diffs = frames[p].sort_values(["day", "timestamp"]).groupby("day")["mid_price"].diff().dropna()
    n     = len(diffs)
    mu    = diffs.mean()             # drift per 100-tick step
    sigma = diffs.std(ddof=1)        # vol per 100-tick step
    se    = sigma / np.sqrt(n)       # SE of the mean
    t     = mu / se
    p_val = 2 * (1 - stats.norm.cdf(abs(t)))

    rows.append({
        "product":         p,
        "n":               n,
        "mu_per_step":     mu,
        "sigma_per_step":  sigma,
        "SE(mu)":          se,
        "t_stat":          t,
        "p_value":         p_val,
        "signal_to_noise": mu / sigma,   # μ/σ — how much drift per unit of vol per step
    })

drift_df = pd.DataFrame(rows)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
drift_df


,product,n,mu_per_step,sigma_per_step,SE(mu),t_stat,p_value,signal_to_noise
0,PEBBLES_XS,29997,-0.1326,15.0524,0.0869,-1.5261,0.1270,-0.0088
1,PEBBLES_S,29997,-0.0651,15.0203,0.0867,-0.7511,0.4526,-0.0043
2,PEBBLES_M,29997,0.0230,15.1336,0.0874,0.2631,0.7925,0.0015
3,PEBBLES_L,29997,-0.0297,15.0255,0.0868,-0.3424,0.7321,-0.0020
4,PEBBLES_XL,29997,0.2046,30.3142,0.1750,1.1687,0.2425,0.0067
